# Course 3: Large Language Models
## Session 2: Tokenization across languages

**Goal:** measure how many tokens a Bangla word costs under different tokenizers, and train a Bangla BPE.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

### 1. Bytes first
UTF-8 stores a basic Latin letter in 1 byte and a Bangla character in 3. A byte-level tokenizer starts from that handicap.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import unicodedata
ENGLISH_LINES = ["I sing songs in Bangla.", "My golden Bengal, I love you.", "Bangla is my mother tongue.", "The sky is very clear today.",
                 "The little girl was playing by the river.", "Father bought fish from the market.", "Mother is cooking rice in the kitchen.",
                 "We all go to school together.", "Water collects in the field when it rains.", "Birds sing in the morning.",
                 "Grandfather tells stories every evening.", "The boy loves reading books.", "Many boats are moving on the river.",
                 "We eat rice cakes in winter.", "Dhaka is the capital of Bangladesh.", "My friend plays football well."]
def per_word(lines, count): return sum(count(l) for l in lines) / sum(len(l.split()) for l in lines)
bn_bytes = per_word(SAMPLE_LINES, lambda l: len(l.encode("utf-8"))); en_bytes = per_word(ENGLISH_LINES, lambda l: len(l.encode("utf-8")))
print(f"bytes per word: Bangla {bn_bytes:.1f} | English {en_bytes:.1f}")
w = "ক্ষুদ্র"
print(w, "is", len(w), "code points and", len(w.encode("utf-8")), "bytes:", [unicodedata.name(c).replace("BENGALI ", "") for c in w])
assert bn_bytes > 2 * en_bytes

### 2. Train BPE on each language
Same algorithm, same vocabulary size, same amount of text (the 16 sentences are translations of each other).

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

In [ ]:
tok_bn, tok_en = train_bpe(SAMPLE_LINES, 120), train_bpe(ENGLISH_LINES, 120)
count = lambda tok: (lambda l: len(tok.encode(l).ids))
print(f"tokens per word with its own BPE: Bangla {per_word(SAMPLE_LINES, count(tok_bn)):.2f} | English {per_word(ENGLISH_LINES, count(tok_en)):.2f}")
print(tok_bn.encode(SAMPLE_LINES[0]).tokens)

### 3. What happens when the tokenizer was built for another language
Many models use a vocabulary learned mostly from English. Simulate that: learn merges on English bytes only, then apply them to Bangla.

In [ ]:
from collections import Counter
def learn_byte_bpe(lines, n_merges):
    seqs = [[bytes([b]) for b in w.encode("utf-8")] for l in lines for w in l.split()]
    merges = []
    for _ in range(n_merges):
        pairs = Counter(p for s in seqs for p in zip(s, s[1:]))
        if not pairs: break
        best = max(pairs, key=pairs.get); merges.append(best)
        seqs = [apply_merge(s, best) for s in seqs]
    return merges

def apply_merge(s, pair):
    out, i = [], 0
    while i < len(s):
        if i + 1 < len(s) and (s[i], s[i + 1]) == pair: out.append(s[i] + s[i + 1]); i += 2
        else: out.append(s[i]); i += 1
    return out

def byte_bpe_count(line, merges):
    total = 0
    for w in line.split():
        s = [bytes([b]) for b in w.encode("utf-8")]
        for m in merges: s = apply_merge(s, m)
        total += len(s)
    return total

english_merges = learn_byte_bpe(ENGLISH_LINES, 150); bangla_merges = learn_byte_bpe(SAMPLE_LINES, 150)
r = {"English text, English merges": per_word(ENGLISH_LINES, lambda l: byte_bpe_count(l, english_merges)),
     "Bangla text, English merges": per_word(SAMPLE_LINES, lambda l: byte_bpe_count(l, english_merges)),
     "Bangla text, Bangla merges": per_word(SAMPLE_LINES, lambda l: byte_bpe_count(l, bangla_merges))}
for k, v in r.items(): print(f"{k:30s} {v:5.2f} tokens per word")
assert r["Bangla text, English merges"] > 3 * r["Bangla text, Bangla merges"]

With merges learned from English, a Bangla word falls apart into single bytes. This is the mechanism behind the 'token tax' measured by Petrov et al. (2023): the same content costs more tokens, so more money, more latency and less room in the context window. Our sample is tiny, so the sizes of these gaps are illustrations, not measurements.

### 4. Round trips and normalization
A tokenizer must return the text unchanged. Bangla has characters with two encodings (for example য় as one code point or as য plus a nukta), so normalize first.

In [ ]:
a, b = "\u09DF", "\u09AF\u09BC"                         # the same letter, written two ways
print("look the same:", a, b, "| equal as strings:", a == b, "| equal after NFC:", unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))
tok_big = train_bpe(SAMPLE_LINES, 300)
ok = sum(tok_big.decode(tok_big.encode(l).ids) == unicodedata.normalize("NFC", l) for l in SAMPLE_LINES)
print(f"round trip exact for {ok} of {len(SAMPLE_LINES)} training sentences")

### 5. Open tokenizers on real text (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install transformers` and downloads from the Hugging Face hub. Some models need you to accept a licence first.
from transformers import AutoTokenizer
names = ["gpt2", "xlm-roberta-base", "google/mt5-small", "csebuetnlp/banglabert", "hishab/titulm-llama-3.2-1b-v2.0"]   # check each name on the hub
bn_text = open("bangla_sample.txt", encoding="utf-8").read()      # TODO: 1,000 Bangla sentences, for example from Bangla Wikipedia
en_text = open("english_sample.txt", encoding="utf-8").read()     # TODO: 1,000 English sentences of similar content
for name in names:
    t = AutoTokenizer.from_pretrained(name)
    bn = len(t(bn_text)["input_ids"]) / len(bn_text.split()); en = len(t(en_text)["input_ids"]) / len(en_text.split())
    print(f"{name:40s} Bangla {bn:.2f}  English {en:.2f}  ratio {bn / en:.2f}")

### TODO
1. Run the cell above and make a table. Which tokenizer is fairest to Bangla?
2. Train BPE with vocabulary sizes 500, 2,000 and 8,000 on a larger Bangla text and plot tokens per word.
3. Find three Bangla words that your tokenizer splits in a way that breaks a conjunct (যুক্তাক্ষর). Does it matter for the model?